# Bucket 3 Inverse Levered ETF NAV Construction Analysis

APLZ-first prototype for bucket-3 (bucket C in `daily_screener.py`).

This notebook:
- reads bucket-3 universe from `daily_screener.py`
- pulls APLZ NAV snapshots and holdings snapshots from Tradr/AXS feeds
- estimates effective daily beta vs APLD and residuals
- exports a Dropbox HTML report in the same style as bucket-2 research


In [ ]:
from __future__ import annotations

import io
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / 'daily_screener.py').exists():
    REPO_ROOT_RUNTIME = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / 'daily_screener.py').exists():
    REPO_ROOT_RUNTIME = NOTEBOOK_DIR.parent
else:
    REPO_ROOT_RUNTIME = Path(r'C:\\Users\\werdn\\Documents\\Investing\\ls-algo')
if str(REPO_ROOT_RUNTIME) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT_RUNTIME))

from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup

from daily_screener import INVERSE_ETF_UNIVERSE, BENCHMARK_MAP

REPO_ROOT = Path(r"C:\Users\werdn\Documents\Investing\ls-algo")
DROPBOX_DIR = Path(r"C:\Users\werdn\Dropbox\Levered ETFs")
REPORT_HTML = DROPBOX_DIR / "Bucket3_Inverse_Levered_ETF_NAV_Construction_Research.html"
REPORT_PDF = DROPBOX_DIR / "Bucket3_Inverse_Levered_ETF_NAV_Construction_Research.pdf"
SNAPSHOT_DIR = REPO_ROOT / "data" / "holdings_snapshots"
APLZ_PAGE = "https://www.tradretfs.com/aplz"

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})


In [ ]:
bucket3_df = pd.DataFrame(INVERSE_ETF_UNIVERSE, columns=["ETF", "Leverage", "Group"])
bucket3_df["UnderlyingProxy"] = bucket3_df["Group"].map(BENCHMARK_MAP)
bucket3_df = bucket3_df.sort_values(["ETF"]).reset_index(drop=True)
print(f"Bucket 3 size from daily_screener.py: {len(bucket3_df)} ETFs")
display(bucket3_df.head(15))


In [ ]:
def _read_csv_url(url: str) -> pd.DataFrame:
    r = session.get(url, timeout=20)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

def fetch_aplz_page_links() -> dict:
    html = session.get(APLZ_PAGE, timeout=20).text
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        txt = a.get_text(" ", strip=True)
        if href.startswith("/"):
            href = "https://www.tradretfs.com" + href
        links.append((txt, href))

    seen = set()
    uniq = []
    for txt, href in links:
        key = (txt, href)
        if key in seen:
            continue
        seen.add(key)
        uniq.append(key)

    literature = [
        (txt, href) for txt, href in uniq
        if any(k in (txt + " " + href).lower() for k in ["prospectus", "sai", "summary"])
    ]
    return {
        "literature_links": literature,
        "nav_endpoint": "https://axsetf.filepoint.live/v2/aplz/nav",
    }

def fetch_nav_snapshot_for_day(d: date) -> pd.Series | None:
    stamp = d.strftime("%m%d%Y")
    url = f"https://axsetf.filepoint.live/assets/data/NSDEAXS2.{stamp}.csv"
    try:
        df = _read_csv_url(url)
    except Exception:
        return None
    row = df[df["Ticker Symbol"].astype(str).str.upper() == "APLZ"]
    if row.empty:
        return None
    out = row.iloc[0].copy()
    out["_source_url"] = url
    return out

def fetch_holdings_snapshot_for_day(d: date) -> pd.DataFrame | None:
    stamp = d.strftime("%Y%m%d")
    url = f"https://axsetf.filepoint.live/assets/data/BBH_AXS_ETF_PVAL_WEB.{stamp}.csv"
    try:
        df = _read_csv_url(url)
    except Exception:
        return None
    out = df[df["ETF Ticker"].astype(str).str.upper() == "APLZ"].copy()
    if out.empty:
        return None
    out["_source_url"] = url
    return out

def fetch_latest_with_lookback(fetcher, max_days: int = 15):
    for i in range(max_days):
        d = date.today() - timedelta(days=i)
        value = fetcher(d)
        if value is not None:
            return d, value
    return None, None


In [ ]:
aplz_info = fetch_aplz_page_links()
nav_date, nav_row = fetch_latest_with_lookback(fetch_nav_snapshot_for_day, max_days=20)
hold_date, holdings_df = fetch_latest_with_lookback(fetch_holdings_snapshot_for_day, max_days=20)

if nav_row is None or holdings_df is None:
    raise RuntimeError("Could not fetch current APLZ NAV/holdings snapshots.")

for c in ["Market Value", "Shares", "Security Price", "Shares Outstanding", "Total Net Assets"]:
    if c in holdings_df.columns:
        holdings_df[c] = pd.to_numeric(holdings_df[c], errors="coerce")

rows = []
for i in range(120):
    d = date.today() - timedelta(days=i)
    r = fetch_nav_snapshot_for_day(d)
    if r is None:
        continue
    rows.append({
        "date": pd.to_datetime(r["Accounting Date"], format="%d-%b-%y", errors="coerce"),
        "nav": pd.to_numeric(r["NAV"], errors="coerce"),
        "close": pd.to_numeric(r["Closing Market Price"], errors="coerce"),
        "premium_discount_pct": pd.to_numeric(r["Premium (Discount)"], errors="coerce"),
        "shares_out": pd.to_numeric(r["Shares Outstanding"], errors="coerce"),
        "tna": pd.to_numeric(r["Total Net Assets"], errors="coerce"),
    })

nav_hist = pd.DataFrame(rows).dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
nav_hist = nav_hist.drop_duplicates(subset=["date"], keep="last")
nav_hist["nav_ret"] = nav_hist["nav"].pct_change()

und = yf.download(
    "APLD",
    start=(nav_hist["date"].min() - pd.Timedelta(days=5)).strftime("%Y-%m-%d"),
    end=(nav_hist["date"].max() + pd.Timedelta(days=2)).strftime("%Y-%m-%d"),
    progress=False,
    auto_adjust=False,
)
und = und.reset_index()[["Date", "Adj Close"]].rename(columns={"Date": "date", "Adj Close": "apld_adj_close"})
und["date"] = pd.to_datetime(und["date"])
und["apld_ret"] = und["apld_adj_close"].pct_change()

model = nav_hist.merge(und[["date", "apld_ret"]], on="date", how="left")
model["pred_nav_ret_minus2x"] = -2.0 * model["apld_ret"]
model["residual_ret"] = model["nav_ret"] - model["pred_nav_ret_minus2x"]
model = model.dropna(subset=["nav_ret", "apld_ret"]).copy()

beta_eff = np.nan
corr = np.nan
if len(model) >= 8:
    beta_eff = np.cov(model["nav_ret"], model["apld_ret"])[0, 1] / np.var(model["apld_ret"])
    corr = model["nav_ret"].corr(model["apld_ret"])

snap_dir = SNAPSHOT_DIR / hold_date.strftime("%Y-%m-%d")
snap_dir.mkdir(parents=True, exist_ok=True)
snap_file = snap_dir / "APLZ.csv"
holdings_df.to_csv(snap_file, index=False)

print("Latest NAV snapshot:", nav_date)
print("Latest holdings snapshot:", hold_date)
print("Saved holdings snapshot:", snap_file)
print(f"Effective beta(nav vs APLD): {beta_eff:.3f}")
print(f"Return corr(nav vs APLD): {corr:.3f}")

display(holdings_df[["Ticker", "Description", "Security Type", "Market Value", "Market Value Weight", "Shares"]])
display(model.tail(10))


In [ ]:
def _fmt_num(x, digits=2):
    if pd.isna(x):
        return "n/a"
    return f"{x:,.{digits}f}"

def _fmt_pct_ratio(x):
    if pd.isna(x):
        return "n/a"
    return f"{x * 100:.2f}%"

top_holdings = holdings_df.copy()
top_holdings["weight_num"] = top_holdings["Market Value Weight"].astype(str).str.replace("%", "", regex=False)
top_holdings["weight_num"] = pd.to_numeric(top_holdings["weight_num"], errors="coerce")
top_holdings = top_holdings.sort_values("weight_num", ascending=False).head(12)

inception_nav = nav_hist["nav"].iloc[0] if not nav_hist.empty else np.nan
latest_nav = nav_hist["nav"].iloc[-1] if not nav_hist.empty else np.nan
latest_close = nav_hist["close"].iloc[-1] if not nav_hist.empty else np.nan
latest_pd = nav_hist["premium_discount_pct"].iloc[-1] if not nav_hist.empty else np.nan
latest_shares = nav_hist["shares_out"].iloc[-1] if not nav_hist.empty else np.nan
nav_change = (latest_nav / inception_nav - 1.0) if (pd.notna(inception_nav) and inception_nav != 0) else np.nan

lit_html_items = []
for txt, href in aplz_info["literature_links"]:
    label = txt if txt else href
    lit_html_items.append(f'<li><a href="{href}">{label}</a></li>')
lit_html = "".join(lit_html_items) if lit_html_items else "<li>No fund literature links parsed from page.</li>"

rows_html = []
for _, r in top_holdings.iterrows():
    ticker = r.get("Ticker", "")
    desc = r.get("Description", "")
    sec_type = r.get("Security Type", "")
    weight = r.get("Market Value Weight", "")
    mv = _fmt_num(r.get("Market Value", np.nan), 0)
    rows_html.append(f"<tr><td>{ticker}</td><td>{desc}</td><td>{sec_type}</td><td>{weight}</td><td>{mv}</td></tr>")

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Bucket 3 Inverse Levered ETF NAV Construction Research</title>
<style>
body {{ font-family: Inter, Arial, sans-serif; max-width: 1100px; margin: 0 auto; padding: 36px; background: #fafbfc; color: #1f2937; }}
h1 {{ color: #0f3460; border-bottom: 3px solid #0f3460; padding-bottom: 8px; }}
h2 {{ color: #1b4f72; margin-top: 28px; border-left: 4px solid #3498db; padding-left: 10px; }}
.callout {{ background: #e8f0fe; border-left: 4px solid #2563eb; padding: 12px 16px; border-radius: 4px; margin: 14px 0; }}
table {{ border-collapse: collapse; width: 100%; margin-top: 12px; font-size: 13px; }}
th {{ background: #0f3460; color: white; text-align: left; padding: 8px; }}
td {{ border-bottom: 1px solid #e5e7eb; padding: 8px; }}
tr:nth-child(even) {{ background: #f8fafc; }}
code {{ background: #eef2ff; padding: 1px 5px; border-radius: 4px; }}
</style>
</head>
<body>
<h1>Bucket 3 Inverse Levered ETF NAV Construction Research</h1>
<p><strong>Ticker:</strong> APLZ &nbsp;|&nbsp; <strong>Generated:</strong> {datetime.now():%Y-%m-%d %H:%M} &nbsp;|&nbsp; <strong>Universe Source:</strong> daily_screener.py Bucket C</p>

<h2>Executive Takeaways (APLZ Prototype)</h2>
<div class="callout">
<ul>
<li>Observed effective daily beta to APLD: <strong>{_fmt_num(beta_eff, 3)}</strong> (target is -2.000)</li>
<li>NAV change over sampled history: <strong>{_fmt_pct_ratio(nav_change)}</strong></li>
<li>Latest NAV: <strong>${_fmt_num(latest_nav, 4)}</strong>, close: <strong>${_fmt_num(latest_close, 2)}</strong>, premium/discount: <strong>{_fmt_num(latest_pd, 2)}%</strong></li>
<li>Latest shares outstanding: <strong>{_fmt_num(latest_shares, 0)}</strong></li>
</ul>
</div>

<h2>Data Sources Used</h2>
<ul>
<li>Fund page: <a href="https://www.tradretfs.com/aplz">https://www.tradretfs.com/aplz</a></li>
<li>NAV feed: <code>https://axsetf.filepoint.live/assets/data/NSDEAXS2.MMDDYYYY.csv</code></li>
<li>Holdings feed: <code>https://axsetf.filepoint.live/assets/data/BBH_AXS_ETF_PVAL_WEB.YYYYMMDD.csv</code></li>
</ul>

<h2>Current Holdings Decomposition</h2>
<p>Holdings indicate swap-heavy construction with offsetting cash and swap legs, consistent with synthetic daily inverse targeting.</p>
<table>
<tr><th>Ticker</th><th>Description</th><th>Security Type</th><th>Weight</th><th>Market Value (USD)</th></tr>
{''.join(rows_html)}
</table>

<h2>Prospectus / Literature Links Parsed</h2>
<ul>
{lit_html}
</ul>

<h2>Reverse-Engineering Notes</h2>
<ol>
<li>Use holdings weights to estimate gross synthetic exposure and net cash buffer each day.</li>
<li>Estimate implied rebalance flow from target notional = -2 * NAV / underlying price sensitivity.</li>
<li>Track residual between observed NAV return and <code>-2 * underlying return</code>; cluster residuals near close for rebalance slippage signals.</li>
<li>Scale framework across all Bucket 3 tickers after validating APLZ data quality.</li>
</ol>

<p style="color:#6b7280;font-size:12px;margin-top:32px;">Internal research prototype. Validate assumptions against full prospectus and intraday tape before trading.</p>
</body></html>"""

REPORT_HTML.write_text(html, encoding="utf-8")
print("Wrote HTML report:", REPORT_HTML)

try:
    import weasyprint
    weasyprint.HTML(string=html, base_url=str(REPORT_HTML.parent)).write_pdf(str(REPORT_PDF))
    print("Wrote PDF report:", REPORT_PDF)
except Exception as e:
    print("PDF export skipped (weasyprint unavailable):", e)
